**Part 2 · LangChain** · step 12 of 25  
[← 11 Agent memory](11_agent_memory.ipynb) · [🏠 Roadmap](../README.md) · [13 Middleware: summarization & PII →](13_middleware_summarization_and_pii.ipynb)

# 12 · Agents with Structured Output

`create_agent(..., response_format=Schema)` makes the agent's **final** answer follow a Pydantic schema.

The result dict then has two useful keys:
- `result["messages"]` — the full conversation (tool calls included)
- `result["structured_response"]` — the parsed Pydantic object

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool


class AgentOutput(BaseModel):
    """Shape of the agent's final answer."""

    result: str
    confidence: float


@tool
def add_two_number(a: int, b: int) -> int:
    """Add two numbers and return their sum."""
    print("I am getting called")
    return a + b

In [ ]:
agent = create_agent(model="gpt-4o", tools=[add_two_number], response_format=AgentOutput)

resp = agent.invoke({"messages": [HumanMessage("what is 2+3")]})
resp["structured_response"]

In [ ]:
print(resp["structured_response"].confidence)
print(resp["messages"][-1].content)  # the raw last message is still available

## Structured output + memory
Combine `response_format` with a checkpointer to get typed answers across a multi-turn chat.

> If you see *"Deserializing unregistered type `__main__.AgentOutput2` from checkpoint"*, it is only a
> warning: the checkpointer stored your custom Pydantic class. Harmless for learning.

In [ ]:
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver


class AgentOutput2(BaseModel):
    """Final answer plus a self-reported confidence label (e.g. "high")."""

    output: str
    confidence: str


agent_2 = create_agent(
    model="gpt-4o",
    tools=[add_two_number],
    checkpointer=InMemorySaver(),
    response_format=AgentOutput2,
)
config_1 = {"configurable": {"thread_id": str(uuid7())}}

In [ ]:
result1 = agent_2.invoke({"messages": [HumanMessage("I am Ujjawal, and I want to learn about India")]}, config=config_1)
result1["structured_response"].output

In [ ]:
result2 = agent_2.invoke({"messages": [HumanMessage("what is my name")]}, config=config_1)
result2["structured_response"].output

---
✅ **Done with 12 Agent structured output.** Next: [13 Middleware: summarization & PII →](13_middleware_summarization_and_pii.ipynb)  
[🏠 Roadmap](../README.md)